# 10 — Spark-driven Maintenance: Compaction, Manifest Rewrite, Orphan Cleanup

Some Iceberg maintenance operations are exposed only as **Spark stored procedures**:
- `rewrite_data_files` — compact small files into larger ones.
- `rewrite_manifests` — consolidate fragmented manifests.
- `remove_orphan_files` — delete data files no longer referenced by any snapshot.

PyIceberg 0.9 doesn't implement these. We use PySpark inside the Jupyter container,
running in `local[*]` mode and pulling Iceberg + Nessie runtimes via `--packages`.

First `getOrCreate()` downloads ~200 MB of jars to `/home/jovyan/.ivy2` — this is
persisted across kernel restarts via the notebook volume mount.

**Prerequisites:** Run 01–09 for a fragmented file layout worth compacting.

In [1]:
import os

import polars as pl
from pyiceberg.catalog.rest import RestCatalog

S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]

# PyIceberg-side: count files before compaction.
py_catalog = RestCatalog(
    name="nessie",
    **{
        "uri": os.environ["NESSIE_URI"],
        "warehouse": f"s3://{WAREHOUSE_BUCKET}/warehouse",
        "s3.endpoint": os.environ["AWS_S3_ENDPOINT"],
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)
py_table = py_catalog.load_table(("demo", "events"))
py_table.refresh()

files_before = pl.from_arrow(py_table.inspect.files())
print(f"Data files before maintenance: {len(files_before)}")

Data files before maintenance: 9


## 1. Build a SparkSession with Iceberg + Nessie runtimes

Spark talks to Nessie's **native** API (`/api/v2`), not the Iceberg REST endpoint —
the Nessie Spark extensions need it for branch/tag DDL.

In [2]:
from pyspark.sql import SparkSession

PACKAGES = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.99.0",
    "software.amazon.awssdk:bundle:2.24.0",
    # hadoop-aws + matching aws-sdk give Spark the 's3a' filesystem.
    # Needed by remove_orphan_files (it scans S3 via Hadoop FS, not via Iceberg FileIO).
    "org.apache.hadoop:hadoop-aws:3.3.4",
])

spark = (
    SparkSession.builder
    .appName("lakehouse-maintenance")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        "org.projectnessie.spark.extensions.NessieSparkSessionExtensions",
    )
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v2")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.warehouse", f"s3://{WAREHOUSE_BUCKET}/warehouse")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", os.environ["AWS_S3_ENDPOINT"])
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.access-key-id", S3_ACCESS_KEY)
    .config("spark.sql.catalog.nessie.s3.secret-access-key", S3_SECRET_KEY)
    # Map both s3 and s3a schemes to S3AFileSystem for remove_orphan_files's S3 scan.
    .config("spark.hadoop.fs.s3.impl",  "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["AWS_S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.jars.ivy", "/home/jovyan/.ivy2")
    .getOrCreate()
)

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ceeb2932-bf61-4e4c-877f-138d8095f250;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central


	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.99.0 in central
	found software.amazon.awssdk#bundle;2.24.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 341ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.99.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	software.amazon.awssdk#bundle;2.24.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   arti

26/05/15 11:06:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.3


## 2. Smoke test — list tables via Spark

In [3]:
spark.sql("SHOW TABLES IN nessie.demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo|   events|      false|
+---------+---------+-----------+



## 3. Compact small files — `rewrite_data_files`

The default `min-input-files=5` skips partitions with fewer files. We lower it
to 2 so the procedure at least *considers* our partitions, and shrink
`target-file-size-bytes` to 1 MB.

Expect a near-zero result here — this lab's table has ~10 tiny Parquet files
spread across ~9 partitions, so most partitions still have just one file and
have nothing to merge with. On a real table with thousands of small files per
partition (typical of streaming ingest), you'd see meaningful consolidation.

In [4]:
spark.sql("""
    CALL nessie.system.rewrite_data_files(
      table => 'demo.events',
      options => map(
        'min-input-files', '2',
        'target-file-size-bytes', '1048576'
      )
    )
""").show(truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|3                         |4                     |7439                 |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



## 4. Rewrite manifests — `rewrite_manifests`

Same logic: default `min-count-to-merge=100` is far too high for demo data,
lower it so consolidation actually fires.

In [5]:
spark.sql("""
    CALL nessie.system.rewrite_manifests(
      table => 'demo.events'
    )
""").show(truncate=False)

26/05/15 11:07:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|5                        |1                    |
+-------------------------+---------------------+



## 5. Remove orphan files (dry run)

Three things to know:
1. The `gc.enabled` table property must be `true` — Iceberg refuses to GC otherwise.
2. The procedure refuses `older_than` < 24h to avoid racing with in-flight writes.
3. **On Nessie, this procedure is dangerous in general.** Files written by past Nessie
   commits (other branches, tags, history) look like "orphans" to Iceberg's snapshot-
   based view, but they're still referenced by Nessie commits. Running this for real
   on a Nessie-backed table can corrupt other branches. The Nessie-correct tool is
   [`nessie-gc`](https://projectnessie.org/nessie-latest/gc/), which is
   reference-aware. Spark will emit a `NessieUtil` warning when you flip `gc.enabled`.

We use `dry_run => true` so Iceberg lists candidates without deleting anything.
Expect 0 candidates in this lab — every file we wrote is still reachable from some
Nessie commit, so by Iceberg's lights nothing is orphaned yet.

In [6]:
spark.sql("""
    ALTER TABLE nessie.demo.events
    SET TBLPROPERTIES ('gc.enabled' = 'true')
""")

result = spark.sql("""
    CALL nessie.system.remove_orphan_files(
      table => 'demo.events',
      dry_run => true
    )
""")

rows = result.collect()
print(f"Orphan candidates (dry run): {len(rows)}")
for r in rows[:10]:
    print(" ", r['orphan_file_location'])

26/05/15 11:07:12 WARN NessieUtil: The Iceberg property 'gc.enabled' and/or 'write.metadata.delete-after-commit.enabled' is enabled on table 'demo.events' in NessieCatalog. This will likely make data in other Nessie branches and tags and in earlier, historical Nessie commits inaccessible. The recommended setting for those properties is 'false'. Use the 'nessie-gc' tool for Nessie reference-aware garbage collection.
26/05/15 11:07:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Orphan candidates (dry run): 0


## 6. File count after maintenance

In [7]:
py_table.refresh()
files_after = pl.from_arrow(py_table.inspect.files())
print(f"Data files after maintenance: {len(files_after)}")
print(f"Net change: {len(files_before)} → {len(files_after)}")

Data files after maintenance: 10
Net change: 9 → 10


## 7. Stop Spark

In [8]:
spark.stop()

---
**Curriculum complete.** See [docs/iceberg/CURRICULUM.md](../docs/iceberg/CURRICULUM.md) for the notebook index and dependency DAG.